# Environment Setup

In [ ]:
import pandas as pd
import numpy as np
import re
from scipy.optimize import linear_sum_assignment
from collections import defaultdict

def _variant_num(v):
    """Extracts numeric ID from 'Variant X' labels."""
    m = re.search(r"Variant\s+(\d+)", str(v))
    return int(m.group(1)) if m else None

def _entropy_from_counts(counts: np.ndarray):
    """Computes Shannon Entropy for feature usage."""
    s = counts.sum()
    if s == 0: return 0.0
    p = counts / s
    p = p[p > 0]
    return float(-(p * np.log2(p)).sum())

def _gini_from_counts(counts: np.ndarray):
    """Computes Gini concentration for feature usage distribution."""
    x = np.sort(counts.astype(float))
    if x.sum() == 0: return 0.0
    n = len(x)
    cum = np.cumsum(x)
    return float((n + 1 - 2 * (cum.sum() / cum[-1])) / n)

# Functions

In [ ]:
# -----------------------------
# Overlap metrics
# -----------------------------
def compute_overlap_metrics(fired_masks: dict, rules: list):
    """
    fired_masks: dict rule_name -> boolean mask (len n_instances)
    rules: list of rule dicts
    Returns: overlap_metrics (dict), per_rule_overlap_df (DataFrame), match_count_per_instance (np.ndarray)
    """
    rule_names = [r["name"] for r in rules]
    masks = np.vstack([fired_masks[rn] for rn in rule_names]).astype(int)  # shape: (n_rules, n_instances)
    match_counts = masks.sum(axis=0)

    n = match_counts.size
    overlap_rate = float(np.mean(match_counts > 1))
    avg_matches = float(np.mean(match_counts))
    max_matches = int(np.max(match_counts))

    overlap_metrics = {
        "overlap_rate_instances_gt1": overlap_rate,
        "avg_rules_firing_per_instance": avg_matches,
        "max_rules_firing_any_instance": max_matches,
        "pct_instances_exactly1": float(np.mean(match_counts == 1)),
        "pct_instances_exactly0": float(np.mean(match_counts == 0)),
    }

    # Per-rule: among instances covered by rule, how many are also covered by >=1 other rule?
    per_rule_rows = []
    for i, r in enumerate(rules):
        rn = r["name"]
        covered = fired_masks[rn]
        n_cov = int(covered.sum())
        if n_cov == 0:
            frac_overlap = np.nan
        else:
            # instances covered by this rule and also by others => match_counts >= 2
            frac_overlap = float(np.mean(match_counts[covered] >= 2))
        per_rule_rows.append({
            "rule": rn,
            "variant": r["variant"],
            "covered": n_cov,
            "overlap_fraction_with_others": frac_overlap,
            "n_conds": len(r["literals"]),
        })

    per_rule_overlap_df = pd.DataFrame(per_rule_rows).sort_values(["variant","rule"])
    return overlap_metrics, per_rule_overlap_df, match_counts

# -----------------------------
# Simplicity (expanded)
# -----------------------------
def compute_simplicity_metrics(rules: list, X: pd.DataFrame):
    n_rules = len(rules)
    lengths = np.array([len(r["literals"]) for r in rules], dtype=int)

    # Feature usage
    feat_counts = {}
    for r in rules:
        for lit in r["literals"]:
            feat_counts[lit["feat"]] = feat_counts.get(lit["feat"], 0) + 1

    used_features = sorted(feat_counts.keys())
    counts = np.array([feat_counts[f] for f in used_features], dtype=int)

    simplicity = {
        "n_rules": int(n_rules),
        "conds_min": int(lengths.min()) if n_rules else 0,
        "conds_max": int(lengths.max()) if n_rules else 0,
        "conds_mean": float(lengths.mean()) if n_rules else 0.0,
        "conds_median": float(np.median(lengths)) if n_rules else 0.0,
        "total_literals": int(lengths.sum()),
        "n_unique_features_in_rules": int(len(used_features)),
        "n_features_in_X": int(X.shape[1]),
        "feature_sparsity_unused_ratio": float(1 - (len(used_features) / X.shape[1])) if X.shape[1] else 0.0,
        "feature_usage_entropy": _entropy_from_counts(counts),
        "feature_usage_gini": _gini_from_counts(counts),
    }
    return simplicity

# -----------------------------
# Stability under perturbations (fixed rule set)
# -----------------------------
def perturb_binary_features(X: pd.DataFrame, flip_prob: float, rng: np.random.Generator):
    """
    Flip binary 0/1 columns with probability flip_prob.
    Non-binary columns are left unchanged.
    """
    Xp = X.copy()
    # detect binary columns (values subset of {0,1})
    bin_cols = []
    for c in Xp.columns:
        vals = pd.unique(Xp[c].dropna())
        if len(vals) <= 2 and set(vals).issubset({0, 1}):
            bin_cols.append(c)

    if not bin_cols or flip_prob <= 0:
        return Xp

    flips = rng.random((len(Xp), len(bin_cols))) < flip_prob
    B = Xp[bin_cols].to_numpy(dtype=int)
    B = np.where(flips, 1 - B, B)
    Xp[bin_cols] = B
    return Xp

def stability_under_perturbation(X: pd.DataFrame, rules: list, n_runs: int = 30, flip_prob: float = 0.02, seed: int = 0):
    """
    Measures stability of rule-based predictions when flipping a small fraction of binary features.
    Returns mean/std of prediction agreement and rule-firing agreement.
    """
    rng = np.random.default_rng(seed)

    # baseline predictions
    y0, fired0 = predict_with_rules(X, rules)
    y0 = np.array(y0, dtype=object)

    rule_names = [r["name"] for r in rules]
    fired0_mat = np.vstack([fired0[rn] for rn in rule_names]).astype(int)  # (n_rules, n_instances)

    pred_agreements = []
    firing_jaccards = []

    for _ in range(n_runs):
        Xp = perturb_binary_features(X, flip_prob=flip_prob, rng=rng)
        y1, fired1 = predict_with_rules(Xp, rules)
        y1 = np.array(y1, dtype=object)

        # prediction agreement
        pred_agreements.append(float(np.mean(y0 == y1)))

        # rule-firing agreement (per-instance Jaccard between fired rule sets)
        fired1_mat = np.vstack([fired1[rn] for rn in rule_names]).astype(int)
        inter = np.sum((fired0_mat == 1) & (fired1_mat == 1), axis=0)
        union = np.sum((fired0_mat == 1) | (fired1_mat == 1), axis=0)
        jac = np.where(union > 0, inter / union, 1.0)  # if both empty => perfectly stable
        firing_jaccards.append(float(np.mean(jac)))

    return {
        "stability_pred_agreement_mean": float(np.mean(pred_agreements)),
        "stability_pred_agreement_std": float(np.std(pred_agreements, ddof=1)) if n_runs > 1 else 0.0,
        "stability_rule_firing_jaccard_mean": float(np.mean(firing_jaccards)),
        "stability_rule_firing_jaccard_std": float(np.std(firing_jaccards, ddof=1)) if n_runs > 1 else 0.0,
        "n_runs": int(n_runs),
        "flip_prob": float(flip_prob),
    }

# -----------------------------
# Optional: Stability across multiple decision tables (content-level)
# -----------------------------
def rule_content_signature(rules: list):
    """
    Represent each rule as a set of literals (feat, thr, sense).
    Returns dict variant -> set(literals)
    """
    sig = {}
    for r in rules:
        lits = set((lit["feat"], float(lit["thr"]), lit["sense"]) for lit in r["literals"])
        sig[r["variant"]] = lits
    return sig

def stability_across_rule_sets(decision_table_paths: list):
    """
    Compute average Jaccard similarity of rule literal sets per variant across multiple runs.
    Requires multiple exported decision tables (different seeds / bootstrap runs).
    """
    sigs = []
    for p in decision_table_paths:
        rs = parse_decision_table_csv(p)
        sigs.append(rule_content_signature(rs))

    variants = sorted(set().union(*[set(s.keys()) for s in sigs]))
    jaccs = []

    for v in variants:
        # pairwise Jaccard across runs
        sets_v = [s.get(v, set()) for s in sigs]
        for i in range(len(sets_v)):
            for j in range(i + 1, len(sets_v)):
                A, B = sets_v[i], sets_v[j]
                inter = len(A & B)
                union = len(A | B)
                jaccs.append(inter / union if union > 0 else 1.0)

    return {
        "content_stability_jaccard_mean": float(np.mean(jaccs)) if jaccs else 1.0,
        "content_stability_jaccard_std": float(np.std(jaccs, ddof=1)) if len(jaccs) > 1 else 0.0,
        "n_tables": int(len(decision_table_paths)),
    }


# -----------------------------
# 1) Parsing decision table CSV
# -----------------------------
def parse_decision_table_csv(path: str):
    df = pd.read_csv(path)

    ca_col = [c for c in df.columns if "Conditions/Actions" in str(c)][0]
    rule_cols = [c for c in df.columns if re.match(r"Rule\s*\d+", str(c).strip())]

    # action rows (Variant k) contain X in the corresponding rule column
    is_action = df[ca_col].astype(str).str.match(r"\s*Variant\s+\d+")
    is_cond = (~is_action) & df[ca_col].notna() & (df[ca_col].astype(str).str.strip() != "")

    # parse condition string like: "feature <= 0.50"
    def split_condition(cond_str):
        s = str(cond_str).strip()
        # supports only "<=" as in your tables
        left, right = s.split("<=")
        feat = left.strip()
        thr = float(right.strip())
        return feat, thr

    rules = []
    for rc in rule_cols:
        # find variant (row with X)
        vr = df.loc[is_action, [ca_col, rc]].copy()
        vr[rc] = vr[rc].astype(str).str.strip()
        hit = vr[vr[rc].eq("X")]
        variant = hit.iloc[0][ca_col].strip() if len(hit) == 1 else None

        # literals for this rule
        cond_part = df.loc[is_cond, [ca_col, rc]].copy()
        cond_part[rc] = cond_part[rc].astype(str).str.strip()

        literals = []
        for _, row in cond_part.iterrows():
            cell = row[rc]
            if cell not in ("T", "F"):
                continue
            feat, thr = split_condition(row[ca_col])
            # T means feat <= thr, F means feat > thr
            literals.append({"feat": feat, "thr": thr, "sense": cell})

        rules.append({"name": str(rc), "variant": variant, "literals": literals})

    return rules

# -----------------------------
# 2) Apply rules to data
# -----------------------------
def rule_matches(X: pd.DataFrame, rule):
    mask = np.ones(len(X), dtype=bool)
    for lit in rule["literals"]:
        feat, thr, sense = lit["feat"], lit["thr"], lit["sense"]
        if feat not in X.columns:
            raise KeyError(f"Feature '{feat}' not found in X columns.")
        if sense == "T":
            mask &= (X[feat].values <= thr)
        else:  # "F"
            mask &= (X[feat].values > thr)
    return mask

def predict_with_rules(X: pd.DataFrame, rules):
    """
    Returns:
      y_pred: predicted variant (string) or None if no rule fires
      fired:  dict rule_name -> boolean mask of matches
    Policy if multiple rules fire for one instance:
      - choose the first rule (you can replace with highest singularity, etc.)
    """
    fired = {}
    y_pred = np.array([None] * len(X), dtype=object)

    for r in rules:
        m = rule_matches(X, r)
        fired[r["name"]] = m

        # assign only where not assigned yet (first-match policy)
        to_assign = m & pd.isna(y_pred)
        y_pred[to_assign] = r["variant"]

    return y_pred, fired

#Metrics

In [ ]:
def accuracy(y_true, y_pred):
    y_true = np.array(y_true, dtype=object)
    y_pred = np.array(y_pred, dtype=object)
    ok = (y_true == y_pred)
    return float(np.mean(ok))

def macro_f1(y_true, y_pred):
    y_true = np.array(y_true, dtype=object)
    y_pred = np.array(y_pred, dtype=object)
    labels = sorted(set(y_true) | set(y_pred) - {None})
    f1s = []
    for lab in labels:
        tp = np.sum((y_pred == lab) & (y_true == lab))
        fp = np.sum((y_pred == lab) & (y_true != lab))
        fn = np.sum((y_pred != lab) & (y_true == lab))
        prec = tp / (tp + fp) if (tp + fp) > 0 else 0.0
        rec  = tp / (tp + fn) if (tp + fn) > 0 else 0.0
        f1   = (2 * prec * rec / (prec + rec)) if (prec + rec) > 0 else 0.0
        f1s.append(f1)
    return float(np.mean(f1s)) if f1s else 0.0

def coverage(y_pred):
    y_pred = np.array(y_pred, dtype=object)
    return float(np.mean(pd.notna(y_pred)))

def per_rule_precision(y_true, fired_masks, rules):
    y_true = np.array(y_true, dtype=object)
    out = []
    for r in rules:
        m = fired_masks[r["name"]]
        denom = np.sum(m)
        if denom == 0:
            prec = np.nan
        else:
            prec = np.sum(y_true[m] == r["variant"]) / denom
        out.append({"rule": r["name"], "variant": r["variant"], "covered": int(denom), "precision": float(prec) if prec==prec else np.nan,
                    "n_conds": len(r["literals"])})
    return pd.DataFrame(out).sort_values(["variant","rule"])

def evaluate_rules_with_label_matching(
    dataset_csv_path: str,
    decision_table_csv_path: str,
    drop_unnamed: bool = True,
    stability_runs: int = 30,
    stability_flip_prob: float = 0.02,
    stability_seed: int = 0,
):
    # Load rules
    rules = parse_decision_table_csv(decision_table_csv_path)

    # Load dataset (X + y last column)
    data = pd.read_csv(dataset_csv_path)
    if drop_unnamed:
        data = _drop_unnamed(data)

    label_col = data.columns[-1]
    y_cluster = data[label_col].values
    X = data.drop(columns=[label_col])

    # Predict with rules
    y_pred, fired = predict_with_rules(X, rules)  # "Variant k"
    y_pred_num = np.array([_variant_num(v) for v in y_pred], dtype=int)

    # Label matching if y is numeric clusters
    if pd.api.types.is_numeric_dtype(pd.Series(y_cluster)):
        y_cluster = pd.Series(y_cluster).astype(int).values

        clusters = sorted(np.unique(y_cluster))
        variants = sorted(np.unique(y_pred_num))

        cl_idx = {c: i for i, c in enumerate(clusters)}
        va_idx = {v: i for i, v in enumerate(variants)}

        M = np.zeros((len(clusters), len(variants)), dtype=int)
        for c, v in zip(y_cluster, y_pred_num):
            M[cl_idx[c], va_idx[v]] += 1

        row_ind, col_ind = linear_sum_assignment(-M)
        mapping = {clusters[r]: variants[c] for r, c in zip(row_ind, col_ind)}
        y = np.array([f"Variant {mapping[c]}" for c in y_cluster], dtype=object)
    else:
        mapping = None
        y = pd.Series(y_cluster).astype(str).str.strip().values

    # Core metrics
    metrics = {
        "dataset": dataset_csv_path,
        "label_col": label_col,
        "cluster_to_variant_mapping": mapping,
        "coverage": coverage(y_pred),
        "accuracy": accuracy(y, y_pred),
        "macro_f1": macro_f1(y, y_pred),
    }

    # Per-rule precision table
    per_rule_df = per_rule_precision(y, fired, rules)

    # Overlap
    overlap_metrics, per_rule_overlap_df, match_counts = compute_overlap_metrics(fired, rules)
    metrics.update(overlap_metrics)

    # Simplicity (expanded)
    metrics.update(compute_simplicity_metrics(rules, X))

    # Stability under perturbations
    metrics.update(
        stability_under_perturbation(
            X, rules,
            n_runs=stability_runs,
            flip_prob=stability_flip_prob,
            seed=stability_seed
        )
    )

    return metrics, per_rule_df, per_rule_overlap_df

#Function to call with different datasets

In [ ]:

rows = []

for name, dataset_csv, table_csv in [
    ("PATH EVENT LOG 1", " PATH DECISION TREE1", "PATH DECISION TABLE 1" ),
    ("PATH EVENT LOG 2", " PATH DECISION TREE 2", "PATH DECISION TABLE 2" ),
    ("PATH EVENT LOG 3", " PATH DECISION TREE 3", "PATH DECISION TABLE 3" ),
]:
    metrics, _, _ = evaluate_rules_with_label_matching(
        dataset_csv_path=dataset_csv,
        decision_table_csv_path=table_csv,
        stability_runs=50,
        stability_flip_prob=0.02,
        stability_seed=42,
    )
    metrics["log"] = name

    # opzionale: rendi il mapping stampabile
    if isinstance(metrics.get("cluster_to_variant_mapping"), dict):
        metrics["cluster_to_variant_mapping"] = str(metrics["cluster_to_variant_mapping"])

    rows.append(metrics)

metrics_df = pd.DataFrame(rows).set_index("log")
metrics_df